# 07 – Lévy Noise Distribution Fitting

Fits **Normal Inverse Gaussian (NIG)** and Gaussian distributions to the
recovered Lévy increments for both temperature and log-price.

### Key fixes vs. original
- Rewritten from R (`ghyp`) to **Python** (`carma_utils.fit_nig_mle`).
- Both **temperature** and **log-price** increments fitted (original had only temp).
- Saves machine-readable `levy_params_temp.json` / `levy_params_price.json`.
- Model comparison table: Gaussian vs NIG (AIC, BIC, log-likelihood).

Inputs:  `data/increments/temp_inc.csv`, `data/increments/price_inc.csv`
Outputs: `results/levy_params_temp.json`, `results/levy_params_price.json`,
         `figures/levy_fit_temp.png`, `figures/levy_fit_price.png`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from config import RES_DIR, FIG_DIR, INCR_DIR
from carma_utils import nig_logpdf, fit_nig_mle

# ── Load recovered increments ─────────────────────────────────────────────────
dL_temp  = pd.read_csv(INCR_DIR / 'temp_inc.csv')['dL_temp'].to_numpy()
dL_price = pd.read_csv(INCR_DIR / 'price_inc.csv')['dL_price'].to_numpy()

print(f'Temperature increments : {len(dL_temp):,} obs')
print(f'Log-price increments   : {len(dL_price):,} obs')

print('\nSummary statistics:')
for label, dL in [('Temperature', dL_temp), ('Log-price', dL_price)]:
    print(f'\n  {label}:')
    print(f'    mean={dL.mean():.6f}  std={dL.std():.6f}')
    print(f'    skew={scipy_stats.skew(dL):.4f}  '
          f'kurt (excess)={scipy_stats.kurtosis(dL, fisher=True):.4f}')
    print(f'    min={dL.min():.4f}  max={dL.max():.4f}')

## 1.  NIG MLE fitting

The NIG(α, β, μ, δ) density is:
$$f(x) = \frac{\alpha\delta K_1(\alpha\sqrt{\delta^2+(x-\mu)^2})}{\pi\sqrt{\delta^2+(x-\mu)^2}}
\exp\!\bigl(\delta\sqrt{\alpha^2-\beta^2}+\beta(x-\mu)\bigr)$$

We fit by maximum likelihood with multi-start Nelder-Mead via `fit_nig_mle()`.
A Gaussian fit is computed for comparison (closed-form MLE: μ̂ = mean, σ̂ = std).

In [ ]:
def gaussian_fit(dL):
    """Closed-form Gaussian MLE and information criteria."""
    mu_g = float(dL.mean())
    sig_g = float(dL.std(ddof=0))
    loglik_g = float(scipy_stats.norm.logpdf(dL, loc=mu_g, scale=sig_g).sum())
    n = len(dL)
    k = 2
    aic_g = -2 * loglik_g + 2 * k
    bic_g = -2 * loglik_g + k * np.log(n)
    return {'mu': mu_g, 'sigma': sig_g, 'loglik': loglik_g,
            'aic': aic_g, 'bic': bic_g, 'n_params': k}

levy_results = {}
comparison_rows = []

for label, dL, key in [
        ('Temperature', dL_temp,  'temperature'),
        ('Log-price',   dL_price, 'logprice'),
    ]:
    print(f'\n{"="*60}')
    print(f'  {label}  (n={len(dL):,})')
    print(f'{"="*60}')

    # Gaussian
    g = gaussian_fit(dL)
    print(f'\nGaussian MLE:')
    print(f'  μ={g["mu"]:.6f}  σ={g["sigma"]:.6f}')
    print(f'  loglik={g["loglik"]:.2f}  AIC={g["aic"]:.2f}  BIC={g["bic"]:.2f}')
    comparison_rows.append({'series': label, 'model': 'Gaussian',
                            'loglik': g['loglik'], 'AIC': g['aic'], 'BIC': g['bic'],
                            'n_params': g['n_params']})

    # NIG (cached)
    nig_path = RES_DIR / f'levy_params_{key}.json'
    if nig_path.exists():
        print(f'\nLoading cached NIG fit …')
        with open(nig_path) as f:
            nig_params = json.load(f)
        nig_pars = nig_params['nig']
    else:
        print(f'\nFitting NIG …')
        nig_pars = fit_nig_mle(dL, verbose=True)

    alpha, beta, mu, delta = nig_pars['alpha'], nig_pars['beta'], nig_pars['mu'], nig_pars['delta']
    loglik_nig = float(nig_logpdf(dL, alpha, beta, mu, delta).sum())
    n = len(dL); k = 4
    aic_nig = -2 * loglik_nig + 2 * k
    bic_nig = -2 * loglik_nig + k * np.log(n)
    print(f'\nNIG MLE:')
    print(f'  α={alpha:.6f}  β={beta:.6f}  μ={mu:.6f}  δ={delta:.6f}')
    print(f'  loglik={loglik_nig:.2f}  AIC={aic_nig:.2f}  BIC={bic_nig:.2f}')
    comparison_rows.append({'series': label, 'model': 'NIG',
                            'loglik': loglik_nig, 'AIC': aic_nig, 'BIC': bic_nig,
                            'n_params': k})

    nig_pars.update({'loglik': loglik_nig, 'aic': aic_nig, 'bic': bic_nig})
    levy_results[key] = {'gaussian': g, 'nig': nig_pars}

    # Save
    with open(nig_path, 'w') as f:
        json.dump(levy_results[key], f, indent=2)
    print(f'  Saved → {nig_path}')

df_compare = pd.DataFrame(comparison_rows)
print('\n\nModel comparison:')
print(df_compare.to_string(index=False, float_format='{:.2f}'.format))

## 2.  Diagnostic plots

For each series:
- **Left**: histogram with Gaussian (blue) and NIG (red) overlaid
- **Right**: QQ-plot against N(0,1) of standardised increments (NIG fit quantiles in red)

In [ ]:
from scipy.stats import norm

def nig_pdf_grid(xx, alpha, beta, mu, delta):
    """Evaluate NIG density on a grid (vectorised)."""
    return np.exp(nig_logpdf(xx, alpha, beta, mu, delta))

for label, dL, key, xlim in [
        ('Temperature', dL_temp,  'temperature', (-3, 3)),
        ('Log-price',   dL_price, 'logprice',    (-6, 6)),
    ]:
    nig_p = levy_results[key]['nig']
    g_p   = levy_results[key]['gaussian']
    alpha, beta, mu, delta = nig_p['alpha'], nig_p['beta'], nig_p['mu'], nig_p['delta']

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{label} Lévy increments', fontsize=12)

    # ── Histogram + fitted densities ─────────────────────────────────────────
    ax = axes[0]
    ax.hist(dL, bins=300, density=True, color='lightgray', edgecolor='none', label='data')
    xx = np.linspace(*xlim, 500)
    ax.plot(xx, norm.pdf(xx, g_p['mu'], g_p['sigma']), 'b-', lw=2, label='Gaussian')
    ax.plot(xx, nig_pdf_grid(xx, alpha, beta, mu, delta), 'r-', lw=2, label='NIG')
    ax.set_xlim(*xlim)
    ax.set_xlabel('increment')
    ax.set_ylabel('density')
    ax.set_title('Density fit')
    ax.legend(fontsize=8)

    # ── Log-density (to see tails) ────────────────────────────────────────────
    ax = axes[1]
    dens, bins_e = np.histogram(dL, bins=300, density=True)
    bin_mid = 0.5 * (bins_e[:-1] + bins_e[1:])
    with np.errstate(divide='ignore'):
        log_dens = np.where(dens > 0, np.log(dens), np.nan)
    ax.scatter(bin_mid, log_dens, s=4, c='gray', alpha=0.6, label='data')
    ax.plot(xx, np.log(norm.pdf(xx, g_p['mu'], g_p['sigma'])), 'b-', lw=2, label='Gaussian')
    ax.plot(xx, nig_logpdf(xx, alpha, beta, mu, delta), 'r-', lw=2, label='NIG')
    ax.set_xlim(*xlim)
    ax.set_xlabel('increment')
    ax.set_ylabel('log density')
    ax.set_title('Log-density (tail behaviour)')
    ax.legend(fontsize=8)

    # ── QQ-plot vs Normal ─────────────────────────────────────────────────────
    ax = axes[2]
    dL_std = (dL - dL.mean()) / dL.std()
    (osm, osr), (slope, intercept, _) = scipy_stats.probplot(dL_std, dist='norm')
    ax.scatter(osm, osr, s=1, alpha=0.3)
    xl = np.array([osm.min(), osm.max()])
    ax.plot(xl, slope*xl + intercept, 'r-', lw=2)
    ax.set_xlabel('Theoretical N(0,1) quantiles')
    ax.set_ylabel('Sample quantiles')
    ax.set_title('QQ-plot (std increments vs Normal)')

    plt.tight_layout()
    plt.savefig(FIG_DIR / f'levy_fit_{key}.png', dpi=150)
    plt.show()
    print(f'Saved → {FIG_DIR}/levy_fit_{key}.png')